# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [20]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
SELECT COUNT(*)
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬────────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │  gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_cl

In [21]:
con.sql(f"""
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
LIMIT 5
""").show()

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬────────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │  gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_cl

In [22]:
# This cell is for CODE (numbers, a query, a check).
print('1. One row represents the daily performance of one content item for one client on one report date.')
print('2. I use fact_content_daily_performance as the main performance table and may join dim_content for content attributes.')
print('3. I use the mid-panel month 2026-03 to develop and validate features, keeping the final month as a sealed test period.')
print('4. I rank content pages by refresh priority score. Pages with declining search visibility or weak engagement signals receive higher priority for editor review.')
print('5. I exclude future performance data and editor decisions after the prediction date because they would cause data leakage.')
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


1. One row represents the daily performance of one content item for one client on one report date.
2. I use fact_content_daily_performance as the main performance table and may join dim_content for content attributes.
3. I use the mid-panel month 2026-03 to develop and validate features, keeping the final month as a sealed test period.
4. I rank content pages by refresh priority score. Pages with declining search visibility or weak engagement signals receive higher priority for editor review.
5. I exclude future performance data and editor decisions after the prediction date because they would cause data leakage.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [26]:
# This cell is for CODE (numbers, a query, a check).
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT report_date || client_hash_id || content_hash_id) AS unique_grain_rows
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03';
""").show()

con.sql("""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03';
""").show()

con.sql("""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
AND gsc_data_available IS TRUE;
""").show()
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────────────────┐
│ total_rows │ unique_grain_rows │
│   int64    │       int64       │
├────────────┼───────────────────┤
│    9841378 │           9841378 │
└────────────┴───────────────────┘

┌───────────┬────────────┬────────────┐
│ row_count │ first_date │ last_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│        3611061 │
└────────────────┘



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [24]:
# This cell is for CODE (numbers, a query, a check).
feature_df = con.sql("""
SELECT
    content_hash_id,
    AVG(gsc_impressions) AS impressions,
    AVG(gsc_clicks) AS clicks,
    AVG(gsc_avg_position) AS avg_position,
    AVG(ga4_sessions) AS sessions,
    AVG(scroll_events) AS scroll_events
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
GROUP BY content_hash_id
""").df()

feature_df.head()
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,impressions,clicks,avg_position,sessions,scroll_events
0,content_b7e512995f79d5a6,36.774194,0.064516,4.394234,0.000000,0.000000
1,content_05597932fe4da067,1.838710,0.000000,2.714744,0.000000,0.000000
2,content_905aa32a0230694e,4.806452,0.000000,6.481453,0.363636,0.000000
3,content_05434271b257bb68,45.838710,0.193548,6.320337,0.818182,0.090909
4,content_d056587ff7faca0c,89.354839,0.516129,4.459107,0.272727,0.000000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [25]:
# This cell is for CODE (numbers, a query, a check).
df = feature_df.copy()

df["refresh_priority"] = (
    (df["avg_position"] > 20) &
    (df["sessions"] < 10)
).astype(int)
df["leak_feature"] = df["refresh_priority"]
df = df.drop(columns=["leak_feature"])
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.